# Investment Buddy
### Investment Recommendation System using Machine Learning

This project builds a machine learning model that recommends the best investment option based on a person's financial profile. We use a real-world style financial dataset, explore it, clean it, engineer features, train models, and finally predict for a new user.

## Part 1 - Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
print('All libraries imported!')

## Part 2 - Load Dataset

Make sure `synthetic_personal_finance_dataset.csv` is in the same folder as this notebook.

In [ ]:
df = pd.read_csv('synthetic_personal_finance_dataset.csv')
print('Dataset loaded successfully!')
print('Shape:', df.shape)
df.head()

## Part 3 - Initial Exploration

In [ ]:
print('Shape:', df.shape)
print()
print('Column Names:')
print(df.columns.tolist())

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print('Missing Values:')
print(df.isnull().sum())

In [ ]:
print('Duplicate Rows:', df.duplicated().sum())

## Part 4 - Exploratory Data Analysis (EDA)

In [ ]:
numerical_cols = ['age', 'monthly_income_usd', 'monthly_expenses_usd',
                  'savings_usd', 'credit_score', 'debt_to_income_ratio',
                  'savings_to_income_ratio']

df[numerical_cols].hist(bins=30, figsize=(14, 10))
plt.suptitle('Distribution of Numerical Columns', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(df[numerical_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
categorical_cols = ['gender', 'education_level', 'employment_status', 'has_loan', 'region']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    sns.countplot(data=df, x=col, ax=axes[i])
    axes[i].set_title(col)
    axes[i].tick_params(axis='x', rotation=30)

axes[-1].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

sns.histplot(df['monthly_income_usd'], bins=40, ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title('Income Distribution')

sns.histplot(df['savings_usd'], bins=40, ax=axes[0, 1], color='green')
axes[0, 1].set_title('Savings Distribution')

sns.histplot(df['credit_score'], bins=40, ax=axes[1, 0], color='orange')
axes[1, 0].set_title('Credit Score Distribution')

sns.histplot(df['age'], bins=30, ax=axes[1, 1], color='purple')
axes[1, 1].set_title('Age Distribution')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x='employment_status', y='monthly_income_usd', ax=axes[0])
axes[0].set_title('Income by Employment Status')
axes[0].tick_params(axis='x', rotation=20)

sns.boxplot(data=df, x='education_level', y='credit_score', ax=axes[1])
axes[1].set_title('Credit Score by Education Level')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

## Part 5 - Data Cleaning

In [ ]:
df["loan_type"] = df["loan_type"].fillna("None")

print("Missing values after cleaning:")
print(df.isnull().sum())
print()
print("Shape:", df.shape)

## Part 6 - Feature Engineering

We create new columns from existing ones to give the model more useful information.

In [ ]:
df['Disposable_Income'] = df['monthly_income_usd'] - df['monthly_expenses_usd']
df['Expense_Ratio'] = df['monthly_expenses_usd'] / df['monthly_income_usd']
df['Financial_Health_Score'] = (df['credit_score'] / 850) * 100

print('New features created!')
df[['Disposable_Income', 'Expense_Ratio', 'Financial_Health_Score']].head()

## Part 7 - Create Target Variable

The dataset does not have an investment recommendation column, so we create one using simple financial rules based on income, credit score, savings ratio, age, and debt.

In [ ]:
def assign_investment(row):
    income = row['monthly_income_usd']
    credit = row['credit_score']
    savings_ratio = row['savings_to_income_ratio']
    age = row['age']
    dti = row['debt_to_income_ratio']

    if credit >= 750 and income >= 6000 and savings_ratio >= 6 and dti <= 0.3:
        return 'Stocks'
    elif age <= 30 and income >= 4500 and credit >= 620 and dti <= 0.5:
        return 'Cryptocurrency'
    elif credit >= 650 and income >= 4000 and dti <= 0.6:
        return 'Mutual Fund'
    elif credit >= 500 and income >= 2000:
        return 'Fixed Deposit'
    else:
        return 'Government Bond'

df['Investment_Recommendation'] = df.apply(assign_investment, axis=1)

print('Target variable created!')
print()
print(df['Investment_Recommendation'].value_counts())

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='Investment_Recommendation',
              order=df['Investment_Recommendation'].value_counts().index)
plt.title('Investment Recommendation Distribution')
plt.xticks(rotation=20)
plt.show()

## Part 8 - Preprocessing

In [ ]:
df.drop(columns=['user_id', 'record_date', 'loan_type'], inplace=True)

le_gender = LabelEncoder()
le_education = LabelEncoder()
le_employment = LabelEncoder()
le_job = LabelEncoder()
le_loan = LabelEncoder()
le_region = LabelEncoder()
target_encoder = LabelEncoder()

df['gender'] = le_gender.fit_transform(df['gender'])
df['education_level'] = le_education.fit_transform(df['education_level'])
df['employment_status'] = le_employment.fit_transform(df['employment_status'])
df['job_title'] = le_job.fit_transform(df['job_title'])
df['has_loan'] = le_loan.fit_transform(df['has_loan'])
df['region'] = le_region.fit_transform(df['region'])
df['Investment_Recommendation'] = target_encoder.fit_transform(df['Investment_Recommendation'])

print('Encoding done!')
print('Classes:', list(target_encoder.classes_))

In [ ]:
X = df.drop('Investment_Recommendation', axis=1)
y = df['Investment_Recommendation']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Train size:', X_train_scaled.shape)
print('Test size:', X_test_scaled.shape)

## Part 9 - Train and Compare Models

We train three models and compare their accuracy on the test set.

In [ ]:
lr = LogisticRegression(max_iter=1000)
dt = DecisionTreeClassifier(random_state=42)
rf = RandomForestClassifier(random_state=42)

lr.fit(X_train_scaled, y_train)
dt.fit(X_train_scaled, y_train)
rf.fit(X_train_scaled, y_train)

lr_acc = accuracy_score(y_test, lr.predict(X_test_scaled))
dt_acc = accuracy_score(y_test, dt.predict(X_test_scaled))
rf_acc = accuracy_score(y_test, rf.predict(X_test_scaled))

results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree', 'Random Forest'],
    'Accuracy': [round(lr_acc, 4), round(dt_acc, 4), round(rf_acc, 4)]
})

print(results.sort_values('Accuracy', ascending=False).to_string(index=False))

## Part 10 - Hyperparameter Tuning with GridSearchCV

We use Grid Search to find the best settings for our Random Forest model. This is one of the key advanced concepts in this project.

In [ ]:
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train_scaled, y_train)

print('Best Parameters:', grid_search.best_params_)
print('Best CV Accuracy:', round(grid_search.best_score_, 4))

best_model = grid_search.best_estimator_
test_acc = accuracy_score(y_test, best_model.predict(X_test_scaled))
print('Test Accuracy:', round(test_acc, 4))

## Part 11 - Model Evaluation

In [ ]:
y_pred = best_model.predict(X_test_scaled)

print('Accuracy:', round(accuracy_score(y_test, y_pred), 4))
print()
print(classification_report(y_test, y_pred, target_names=target_encoder.classes_))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_encoder.classes_,
            yticklabels=target_encoder.classes_)
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

In [ ]:
feature_names = X.columns.tolist()
importances = best_model.feature_importances_

feat_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feat_df = feat_df.sort_values('Importance', ascending=False).head(10)

plt.figure(figsize=(9, 5))
sns.barplot(data=feat_df, x='Importance', y='Feature')
plt.title('Top 10 Feature Importances')
plt.show()

## Part 12 - Save the Model

In [ ]:
joblib.dump(best_model, 'investment_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(target_encoder, 'target_encoder.pkl')

label_encoders = {
    'gender': le_gender,
    'education_level': le_education,
    'employment_status': le_employment,
    'job_title': le_job,
    'has_loan': le_loan,
    'region': le_region
}
joblib.dump(label_encoders, 'label_encoders.pkl')

print('Model saved: investment_model.pkl')
print('Scaler saved: scaler.pkl')
print('Encoders saved: target_encoder.pkl, label_encoders.pkl')

## Part 13 - Personalized Investment Recommendation

Run this section after all previous cells. Enter your financial details and the model will tell you the best investment option for your profile.

In [ ]:
print('=' * 50)
print('   INVESTMENT BUDDY - Get Your Recommendation   ')
print('=' * 50)
print()

age        = int(input('Age: '))
income     = float(input('Monthly Income (USD): '))
expenses   = float(input('Monthly Expenses (USD): '))
savings    = float(input('Total Savings (USD): '))
credit     = int(input('Credit Score (300-850): '))
loan_amt   = float(input('Loan Amount (USD, enter 0 if no loan): '))
dti        = float(input('Debt to Income Ratio (e.g. 0.3, enter 0 if no loan): '))
gender     = input('Gender (Male / Female / Other): ').strip()
education  = input('Education (High School / Associate / Bachelor / Master / PhD): ').strip()
employment = input('Employment (Employed / Self-employed / Unemployed / Student): ').strip()
job        = input('Job Title (Doctor / Engineer / Teacher / Manager / Driver / Salesperson / Artist / Nurse / Student): ').strip()
loan       = input('Have a Loan? (Yes / No): ').strip()
region     = input('Region (North America / Europe / Asia / Africa / Other): ').strip()

print()
print('Analysing your profile...')

In [ ]:
savings_ratio  = savings / (income * 12) if income > 0 else 0
disposable     = income - expenses
expense_ratio  = expenses / income if income > 0 else 0
health_score   = (credit / 850) * 100

gender_enc     = le_gender.transform([gender])[0]     if gender     in le_gender.classes_     else 0
education_enc  = le_education.transform([education])[0]  if education  in le_education.classes_  else 0
employment_enc = le_employment.transform([employment])[0] if employment in le_employment.classes_ else 0
job_enc        = le_job.transform([job])[0]           if job        in le_job.classes_        else 0
loan_enc       = le_loan.transform([loan])[0]         if loan       in le_loan.classes_       else 0
region_enc     = le_region.transform([region])[0]     if region     in le_region.classes_     else 0

new_user = pd.DataFrame([{
    'age':                     age,
    'gender':                  gender_enc,
    'education_level':         education_enc,
    'employment_status':       employment_enc,
    'job_title':               job_enc,
    'monthly_income_usd':      income,
    'monthly_expenses_usd':    expenses,
    'savings_usd':             savings,
    'has_loan':                loan_enc,
    'loan_amount_usd':         loan_amt,
    'loan_term_months':        0,
    'monthly_emi_usd':         0,
    'loan_interest_rate_pct':  0,
    'debt_to_income_ratio':    dti,
    'credit_score':            credit,
    'savings_to_income_ratio': savings_ratio,
    'region':                  region_enc,
    'Disposable_Income':       disposable,
    'Expense_Ratio':           expense_ratio,
    'Financial_Health_Score':  health_score
}])

new_user_scaled = scaler.transform(new_user)

prediction    = best_model.predict(new_user_scaled)
probabilities = best_model.predict_proba(new_user_scaled)[0]

result     = target_encoder.inverse_transform(prediction)[0]
confidence = round(max(probabilities) * 100, 2)

return_map = {
    'Stocks':          '10% - 15% per year',
    'Cryptocurrency':  '20% - 60% per year (high risk)',
    'Mutual Fund':     '7%  - 11% per year',
    'Fixed Deposit':   '4%  -  7% per year',
    'Government Bond': '5%  -  8% per year'
}

reason_map = {
    'Stocks':          'Strong income, high savings and good credit score make you suitable for equity markets.',
    'Cryptocurrency':  'Young age with decent income and credit supports high-risk, high-reward digital assets.',
    'Mutual Fund':     'Your balanced profile suits professionally managed diversified funds.',
    'Fixed Deposit':   'Your profile suggests safe, fixed-return investments for capital preservation.',
    'Government Bond': 'Low-risk government securities are the safest option for your current profile.'
}

print('\n' + '=' * 50)
print('      YOUR INVESTMENT RECOMMENDATION')
print('=' * 50)
print(f'  Recommended Investment : {result}')
print(f'  Confidence Score       : {confidence}%')
print(f'  Expected Returns       : {return_map.get(result, "N/A")}')
print('-' * 50)
print('  Why this recommendation?')
print(f'  {reason_map.get(result, "Based on your financial profile.")}')
print('=' * 50)

all_labels = target_encoder.inverse_transform(range(len(probabilities)))
prob_df = pd.DataFrame({
    'Investment': all_labels,
    'Probability (%)': (probabilities * 100).round(2)
}).sort_values('Probability (%)', ascending=False).reset_index(drop=True)

print()
print('Probability Breakdown:')
print(prob_df.to_string(index=False))

colors = ['#1565C0' if inv == result else '#B0BEC5' for inv in prob_df['Investment']]
plt.figure(figsize=(8, 4))
plt.bar(prob_df['Investment'], prob_df['Probability (%)'], color=colors)
plt.title(f'Prediction Confidence - Recommended: {result}', fontsize=13)
plt.ylabel('Probability (%)')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()